In [1]:
#import neessary libraries for feature engineering
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix,hstack

In [2]:
#load the processed datasets
training = pd.read_csv(r'C:\Users\GIGABYTE\Desktop\drug-sentiment-analysis\data\processed\training_data.csv')
test = pd.read_csv(r'C:\Users\GIGABYTE\Desktop\drug-sentiment-analysis\data\processed\test_data.csv')

In [3]:
training.head()

,unique_hash,text,drug,sentiment,cleaned_text,clean_drug
0,2e180be4c9214c1f5ab51fd8cc32bc80c9f612e0,Autoimmune diseases tend to come in clusters. ...,gilenya,2,autoimmune disease tend come cluster gilenya f...,gilenya
1,9eba8f80e7e20f3a2f48685530748fbfa95943e4,I can completely understand why you’d want to ...,gilenya,2,completely understand youd want try result rep...,gilenya
2,fe809672251f6bd0d986e00380f48d047c7e7b76,Interesting that it only targets S1P-1/5 recep...,fingolimod,2,interesting target sp receptor rather like fin...,fingolimod
3,bd22104dfa9ec80db4099523e03fae7a52735eb6,"Very interesting, grand merci. Now I wonder wh...",ocrevus,2,interesting grand merci wonder lemtrada ocrevu...,ocrevus
4,b227688381f9b25e5b65109dd00f7f895e838249,"Hi everybody, My latest MRI results for Brain ...",gilenya,1,hi everybody latest mri result brain cervical ...,gilenya


In [4]:
# check number of unique words in the cleaned_text column
unique_words = set(" ".join(training['cleaned_text']).split())

print("Vocabulary size:", len(unique_words))

Vocabulary size: 40035


In [5]:
from collections import Counter

all_words = " ".join(training['cleaned_text']).split()

word_counts = Counter(all_words)

rare_words = [word for word, count in word_counts.items() if count == 1]

print("Rare words:", len(rare_words))

Rare words: 17396


# Observation

The dataset contains 40,035 unique words, out of which 17,396 words appear only once across all reviews. This indicates a high proportion of rare tokens, which is common in natural language datasets due to spelling variations, domain-specific terminology, and noise.

To reduce dimensionality and improve model generalization, rare words will be filtered during TF-IDF feature extraction using a minimum document frequency threshold (min_df).

In [6]:
# vectorizer
tfidf = TfidfVectorizer(
    max_features=6000,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.9
)

In [ ]:
# checking rare drugs 
drug_counts = training['clean_drug'].value_counts()

rare_drugs = drug_counts[drug_counts < 5].index

In [20]:
rare_drugs

Index(['aubagio', 'photodynamic therapy', 'cyltezo', 'necitumumab', 'mekinist',
       'dexamethasone implant', 'remsima', 'certolizumab pegol', 'aflibercept',
       'macugen', 'arzerra', 'zykadia', 'tafinlar', 'ceritinib', 'ixifi',
       'brolucizumab', 'amjevita', 'teriflunomide', 'trametinib', 'lorlatinib',
       'dabrafenib', 'pan-retinal photocoagulation', 'infliximab-dyyb',
       'geftinib', 'pf-00547659', 'alunbrig', 'pemetrexed disodium',
       'portrazza', 'movectro', 'brigatinib', 'pegaptanib', 'alemtuzumab',
       'rhumab 2h7', 'filgotinib', 'alectnib', 'ct-p13', 'guselkumab',
       'cyramza'],
      dtype='object', name='clean_drug')

In [ ]:
# repalcing rare drugs with "other drugs"
training['clean_drug'] = training['clean_drug'].replace(rare_drugs, 'other_drug')
test['clean_drug'] = test['clean_drug'].replace(rare_drugs, 'other_drug')

In [9]:
#get only the needed columns for feature engineering
train = training[["cleaned_text", "clean_drug", "sentiment"]]
testing = test[["cleaned_text", "clean_drug"]]

In [10]:
# define feature and target variables
X_text = training['cleaned_text']
X_drug = training['clean_drug']
y = training['sentiment']

In [11]:
#vectorize the text data
X_tfidf = tfidf.fit_transform(X_text)

In [12]:
#one-hot encoding the drug
X_drug_encoded = pd.get_dummies(X_drug, prefix='drug')

In [13]:
# convert one-hot encoding to sparese marix

X_drug_sparse = csr_matrix(X_drug_encoded.values)

In [14]:
# combine the text and drug features

X = hstack([X_tfidf, X_drug_sparse])

In [15]:
X.shape

(5279, 6060)

In [16]:
y.shape

(5279,)

In [17]:
#spliting the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [18]:
#feature engineering for the test dataset
testing_text = testing['cleaned_text']
testing_drug = testing['clean_drug']

testing_tfidf = tfidf.transform(testing_text)
testing_drug_encoded = pd.get_dummies(testing_drug, prefix='drug')
testing_drug_sparse = csr_matrix(testing_drug_encoded.values)
final_testing = hstack([testing_tfidf, testing_drug_sparse])

In [19]:
import joblib

# Save training features and target
joblib.dump(X_train, r"C:\Users\GIGABYTE\Desktop\drug-sentiment-analysis\data\processed/X_train.pkl")
joblib.dump(X_val, r"C:\Users\GIGABYTE\Desktop\drug-sentiment-analysis\data\processed/X_val.pkl")
joblib.dump(y_train, r"C:\Users\GIGABYTE\Desktop\drug-sentiment-analysis\data\processed/y_train.pkl")
joblib.dump(y_val, r"C:\Users\GIGABYTE\Desktop\drug-sentiment-analysis\data\processed/y_val.pkl")

# Save test features
joblib.dump(final_testing, r"C:\Users\GIGABYTE\Desktop\drug-sentiment-analysis\data\processed/X_test.pkl")
joblib.dump(tfidf, r"C:\Users\GIGABYTE\Desktop\drug-sentiment-analysis\data\processed/tfidf_vectorizer.pkl")
joblib.dump(X_drug_encoded.columns, r"C:\Users\GIGABYTE\Desktop\drug-sentiment-analysis\data\processed/drug_columns.pkl")

['C:\\Users\\GIGABYTE\\Desktop\\drug-sentiment-analysis\\data\\processed/drug_columns.pkl']